# Pinecone Semantic Search & Reranking Challenge

A complete end-to-end solution for document reranking and serverless medical note indexing with Pinecone.

---

Part 1: Load Documents & Execute Reranking Model

1. Install Pinecone libraries

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

### 2. Authenticate with Pinecone

In [ ]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

3. Instantiate the Pinecone client

In [ ]:
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

print("✓ Pinecone client instantiated successfully")

4. Define query & documents (Apple example)

In [ ]:
# Query about Apple (company)
query = "Tell me about Apple's products"

# Mix of documents about Apple (fruit) and Apple (company)
documents = [
    "An apple is a crispy, sweet red fruit that grows on apple trees. Apples are rich in fiber and vitamin C, making them a healthy snack choice.",
    "Apple Inc. is a technology company that manufactures iPhones, iPads, MacBooks, and other electronic devices. Founded in 1976, Apple is one of the world's most valuable companies.",
    "The apple tree is a deciduous tree that belongs to the Rosaceae family. It blooms with beautiful white and pink flowers in spring and produces fruit in late summer and fall.",
    "Apple's ecosystem includes iOS, macOS, watchOS, and tvOS operating systems. Users can seamlessly sync their devices using iCloud cloud storage and services.",
    "Granny Smith apples are a popular green variety known for their tart taste and firm texture. They are excellent for baking pies, sauces, and other culinary applications."
]

print(f"Query: {query}")
print(f"\nDocuments to rerank: {len(documents)}")
for i, doc in enumerate(documents, 1):
    print(f"  {i}. {doc[:60]}...")

5. Call the reranker

In [ ]:
# Call the reranker with top_n=3
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

print("✓ Reranking completed")

### 6. Inspect reranked results

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    print(f"\nTop Reranked Results:")
    print("=" * 80)
    for i, m in enumerate(matches):
        print(f"\n{str(i+1).rjust(2)}. ID: {m.document.id} | Score: {m.score:.4f}")
        print(f"   Text: {m.document.text}")

show_reranked_results(query, reranked.data)

---
Part 2: Setup a Serverless Index for Medical Notes

1. Install data & model libraries

In [ ]:
!pip install pandas torch transformers

2. Import modules & define environment settings

In [ ]:
import time
import pandas as pd
from pinecone import ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Cloud and region settings (defaults for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index'

print(f"✓ Cloud: {cloud}")
print(f"✓ Region: {region}")
print(f"✓ Index Name: {index_name}")

3. Create or recreate the index

In [ ]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    print(f"Deleting existing index: {index_name}")
    pc.delete_index(name=index_name)
    time.sleep(2)

# Create a new index
print(f"Creating index: {index_name}")
pc.create_index(
    name=index_name,
    dimension=384,  # all-MiniLM-L6-v2 outputs 384-dimensional vectors
    metric='cosine',  # Cosine similarity for text embeddings
    spec=spec
)

print(f"✓ Index '{index_name}' created successfully")

---
Part 3: Load the Sample Data

1. Download & read JSONL

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from GitHub
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    print(f"Downloading from: {url}")
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)
    print(f"✓ Data downloaded and loaded")

2. Preview the DataFrame

In [ ]:
# Show shape of the DataFrame
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

---
Part 4: Upsert Data into the Index

1. Instantiate index client & upsert

In [ ]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
print(f"Upserting {len(df)} vectors into index...")
index.upsert_from_dataframe(df)
print(f"✓ Upsert completed")

2. Wait for availability

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: {vector_count}")
    return vector_count > 0  # Check if at least 1 vector exists

# Wait until index is ready
print("Waiting for index to be ready...")
max_retries = 30
retry_count = 0

while not is_fresh(index) and retry_count < max_retries:
    time.sleep(2)
    retry_count += 1

if retry_count == max_retries:
    print("Timeout waiting for index")
else:
    print("\n✓ Index ready!")
    stats = index.describe_index_stats()
    print(f"Total vectors: {stats.total_vector_count}")

---
## Part 5: Query & Embedding Function

1. Define embedding function

In [ ]:
def get_embedding(input_question):
    """Generate embedding using sentence-transformers all-MiniLM-L6-v2"""
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    # Tokenize input
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    
    # Generate embeddings
    with torch.no_grad():
        model_output = model(**encoded_input)
    
    # Average over sequence length dimension (dim=1) to get single vector
    embedding = model_output.last_hidden_state[0].mean(dim=0)
    
    return embedding

print("✓ Embedding function defined")

2. Run a semantic search query

In [ ]:
# Build a query to search
question = "patient has chest pain"

print(f"Embedding question: '{question}'")
query_embedding = get_embedding(question).tolist()
print(f"✓ Embedding generated (dimension: {len(query_embedding)})")

Get results
print(f"\nQuerying index for top 10 results...")
results = index.query(vector=query_embedding, top_k=10, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

print(f"✓ Query completed, found {len(sorted_matches)} results")

---
Part 6: Display & Rerank Clinical Notes

1. Display initial search results

In [ ]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nInitial Search Results (by Vector Similarity):')  
    print("=" * 80)
    for i, match in enumerate(matches[:5], 1):  # Show top 5
        print(f'\n{str(i).rjust(2)}. ID: {match["id"]}')
        print(f'   Vector Score: {match["score"]:.4f}')
        if 'metadata' in match:
            metadata = match['metadata']
            print(f'   Metadata: {metadata}')

show_results(question, sorted_matches)

2. Prepare documents for reranking

In [ ]:
# Create documents with concatenated metadata field for reranking
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()]) if match.get('metadata') else 'No metadata'
    }
    for match in results['matches']
]

print(f"✓ Prepared {len(transformed_documents)} documents for reranking")
print(f"\nSample document for reranking:")
print(f"  ID: {transformed_documents[0]['id']}")
print(f"  Reranking Field: {transformed_documents[0]['reranking_field'][:100]}...")

### 3. Execute serverless reranking

In [ ]:
# Define a more specific query for reranking
refined_query = "patient needs cardiac assessment for chest pain and shortness of breath"

print(f"Refined Query: '{refined_query}'")
print(f"\nPerforming reranking...")

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,  # Return top 3 reranked results
    return_documents=True,
)

print(f"✓ Reranking completed, {len(reranked_results.data)} results returned")

4. Show reranked results

In [ ]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results (by LLM Relevance):')  
    print("=" * 80)
    for i, match in enumerate(matches, 1):
        print(f'\n{str(i).rjust(2)}. ID: {match.document.id}')
        print(f'   Reranking Score: {match.score:.4f}')
        print(f'   Reranking Field: {match.document.reranking_field[:100]}...')

show_reranked_results(refined_query, reranked_results.data)

5. Comparison: Vector Search vs. Reranking

In [ ]:
print("COMPARISON: Vector Search vs. Reranking")
print("=" * 80)

print(f"\n📊 Initial Vector Search Results (Top 3):")
for i, match in enumerate(sorted_matches[:3], 1):
    print(f"  {i}. ID: {match['id']:<8} | Vector Score: {match['score']:.4f}")

print(f"\n🎯 After Reranking (Top 3):")
for i, match in enumerate(reranked_results.data, 1):
    print(f"  {i}. ID: {match.document.id:<8} | Rerank Score: {match.score:.4f}")

print("\n💡 Key Insight: Reranking uses LLM-based semantic understanding to improve relevance ordering!")

### 6. Clean up (optional)

In [ ]:
# Delete the index to save resources
print(f"Deleting index: {index_name}")
pc.delete_index(name=index_name)
print("✓ Index deleted successfully")

---

Success Criteria

This notebook demonstrates:

Successfully authenticate with Pinecone  
Run basic document reranking with sensible results  
Create and populate a serverless index with medical notes  
Execute semantic search queries on medical data  
Compare original search results with reranked results  
Understand how reranking improves search relevance using LLM-based scoring

---

Troubleshooting

| Problem | Solution |
|---------|----------|
| **Authentication failed** | Check your API key in the Pinecone dashboard |
| **Index already exists** | The code handles this by deleting and recreating |
| **Dimension mismatch** | Using dimension=384 for all-MiniLM-L6-v2 |
| **No results found** | Wait a few moments after upserting |
| **Model download slow** | Normal on first run - model is being downloaded |
